# saeflow benchmarks on a Colab T4

Runtime -> Change runtime type -> **T4 GPU**, then run all cells. The T4 has no bf16 tensor cores,
so the sweep runs fp32 and fp16. The 65k-width dense baselines at large token counts may
run out of memory on 16 GB; those rows are skipped and logged.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/iamartyaa/saeflow
%cd saeflow
!pip install -q -e ".[test,bench]"

Correctness first: the same tests CI runs, now on the compiled GPU kernels.

In [ ]:
!pytest -q

Encoder: fused encode vs torch vs SAELens, plus the full forward pass at L0=64.

In [ ]:
!python bench/bench_encode.py --tokens 1024,4096,16384

Decoder: sparse decode vs dense torch, torch CSR and SAELens, plus dense -> CSR conversion.

In [ ]:
!python bench/bench_decode.py --tokens 1024,4096,16384,65536 --l0 32,64,128

In [ ]:
!python bench/plot.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("bench/results/*.png")):
    print(p); display(Image(p))

Download the CSVs and plots to commit them under `bench/results/`.

In [ ]:
!zip -qr results.zip bench/results
from google.colab import files
files.download("results.zip")